# 数据驱动决策（ACCA 先修课）· Decision Making with Data
## 第 8 周：长期决策与投资评估

**授课教师**：王奇 副教授（四川大学商学院）  
**研究方向**：人工智能与公司金融  
**邮箱**：qiwangphd@scu.edu.cn

---

### 本周学习目标
1. 掌握货币时间价值：复利、EAR、折现与年金公式
2. 掌握四大投资评估方法：NPV、IRR、ARR、回收期
3. 手动实现 NPV/IRR（二分法 + 线性插值）与回收期计算
4. 进行多项目对比、NPV Profile 与敏感性分析

In [ ]:
import numpy as np                                # 数值计算
import pandas as pd                               # 结果整理
import matplotlib.pyplot as plt                   # 绘图

plt.rcParams['font.sans-serif'] = ['Songti SC', 'SimHei', 'PingFang SC']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False        # 负号显示

## 1. 货币时间价值：复利、折现与年金

| 公式 | 表达式 | 用途 |
|---|---|---|
| 复利终值 | $FV = P(1+r)^n$ | 今天的钱到未来值多少 |
| 实际年利率 | $EAR = (1+r/m)^m - 1$ | 名义利率按 m 次复利 |
| 折现现值 | $PV = FV/(1+r)^n$ | 未来的钱折回今天 |
| 年金现值 | $PV = PMT \cdot \frac{1-(1+r)^{-n}}{r}$ | 每期等额现金流的现值 |

In [ ]:
# ===== 基本计算 =====
P, r, n = 50000, 0.05, 10                         # 本金 5 万、年利率 5%、10 年
FV = P * (1 + r) ** n                             # 复利终值
print(f'(1) 复利终值：50,000 元存 10 年 → {FV:,.0f} 元')   # ≈ 81,445

PV = 100000 / (1.06) ** 3                         # 3 年后 10 万的现值（r=6%）
print(f'(2) 折现现值：3 年后 100,000 元 → 今天 {PV:,.0f} 元')   # ≈ 83,962

PMT, r3, n3 = 20000, 0.08, 5                      # 每年末 2 万、折现率 8%、共 5 年
PV_annuity = PMT * (1 - (1 + r3) ** (-n3)) / r3   # 年金现值公式
print(f'(3) 年金现值：5 年每年 2 万 → {PV_annuity:,.0f} 元')   # ≈ 79,854

EAR = (1 + 0.06 / 12) ** 12 - 1                   # 名义 6% 按月复利的实际年利率
print(f'(4) EAR：6%按月复利 → 实际 {EAR:.2%}（高于名义 6%）')   # 6.17%

In [ ]:
# ===== 可视化：单利 vs 复利的差距 =====
years = np.arange(0, 21)                          # 0~20 年
compound = 10000 * 1.06 ** years                  # 复利增长曲线
simple = 10000 * (1 + 0.06 * years)               # 单利增长直线

plt.figure(figsize=(10, 6))                        # 画布
plt.plot(years, compound, color='#C49A6C', linewidth=2.5, label='复利')   # 复利线
plt.plot(years, simple, '--', color='#8B6F4E', linewidth=2.5, label='单利')   # 单利线
plt.fill_between(years, simple, compound, alpha=0.15, color='#C49A6C')   # 差距区域填色
plt.xlabel('年数')                                 # x 轴
plt.ylabel('金额（元）')                            # y 轴
plt.title('单利 vs 复利（本金 1 万、利率 6%）')      # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.show()                                        # 显示

## 2. NPV：手动实现（黄金标准）

$$NPV = \sum_{t=1}^{n}\frac{CF_t}{(1+r)^t} - I_0$$

In [ ]:
def calculate_npv(cash_flows, discount_rate):
    """手动计算 NPV
    cash_flows: 现金流列表，第 0 期为初始投资（负值）
    discount_rate: 折现率 r"""
    npv = 0                                        # 累加器初始化
    for t, cf in enumerate(cash_flows):            # t=0,1,...,n（enumerate 带序号遍历）
        npv += cf / (1 + discount_rate) ** t       # 每期现金流折现到第 0 期并累加
    return npv                                     # 返回净现值

cf_project = [-100, 30, 40, 35, 25]               # 课堂例题：第0年投资100，1~4年回收（万元）
npv_10 = calculate_npv(cf_project, 0.10)          # 折现率 10%
print(f'NPV(r=10%) = {npv_10:.2f} 万元')           # ≈ +3.70 > 0 → 接受

## 3. IRR：二分法精确求解 + 线性插值估算

IRR 是使 $NPV = 0$ 的折现率。高次方程无解析解，用数值方法求解。

In [ ]:
def calculate_irr_bisection(cash_flows, r_low=0.0, r_high=1.0, tol=1e-8):
    """二分法求 IRR：不断折半缩小区间"""
    if calculate_npv(cash_flows, r_low) * calculate_npv(cash_flows, r_high) > 0:   # 两端同号
        raise ValueError('请调整 r_low 和 r_high 的范围')   # 无法夹住零点
    while abs(r_high - r_low) > tol:               # 区间宽度大于容差就继续
        r_mid = (r_low + r_high) / 2               # 取区间中点
        if calculate_npv(cash_flows, r_mid) > 0:   # 中点 NPV 为正
            r_low = r_mid                          # 真实 IRR 在右半边
        else:                                      # 中点 NPV 为负
            r_high = r_mid                         # 真实 IRR 在左半边
    return (r_low + r_high) / 2                    # 返回最终区间中点

irr_exact = calculate_irr_bisection(cf_project, 0.0, 0.50)   # 二分法精确 IRR
print(f'二分法 IRR = {irr_exact:.4%}')              # ≈ 11.45%

# ===== 线性插值近似（考试常用）=====
npv_15 = calculate_npv(cf_project, 0.15)           # r=15% 的 NPV（负值）
irr_linear = 0.10 + npv_10 / (npv_10 - npv_15) * (0.15 - 0.10)   # 插值公式：r1 + NPV1/(NPV1-NPV2)×(r2-r1)
print(f'线性插值 IRR = {irr_linear:.4%}（10%~15% 区间）')   # ≈ 11.5%，与精确值接近

## 4. ARR 与回收期

In [ ]:
# ===== ARR（会计收益率，用利润不用现金流）=====
I0 = 100                                           # 初始投资 100 万
depreciation = I0 / 4                              # 直线法折旧：100/4 = 25 万/年
profits = [cf - depreciation for cf in cf_project[1:]]   # 各年利润 = 现金流 - 折旧
avg_profit = np.mean(profits)                      # 平均年利润 = 7.5 万
avg_investment = (I0 + 0) / 2                      # 平均投资 = (100+残值0)/2 = 50 万
arr = avg_profit / avg_investment                  # ARR = 15%

print(f'各年利润: {profits}')                        # [5, 15, 10, 0]
print(f'平均利润 {avg_profit} 万 / 平均投资 {avg_investment} 万 → ARR = {arr:.0%}')   # 15%

In [ ]:
def calculate_payback(cash_flows):
    """非折现回收期：累计现金流覆盖初始投资的时间"""
    initial = -cash_flows[0]                       # 初始投资额（取正数）
    cumulative = 0                                 # 累计现金流
    for t in range(1, len(cash_flows)):            # 从第 1 年开始累加
        prev_cum = cumulative                      # 记录年初累计
        cumulative += cash_flows[t]                # 加上当年现金流
        if cumulative >= initial:                  # 本年内回本
            fraction = (initial - prev_cum) / cash_flows[t]   # 年内插值分数
            return (t - 1) + fraction              # 整年数 + 分数年
    return None                                    # 无法回本

def discounted_payback(cash_flows, r):
    """折现回收期：用折现后现金流计算"""
    discounted = [cf / (1 + r) ** t for t, cf in enumerate(cash_flows)]   # 全部折现
    return calculate_payback(discounted)           # 复用非折现函数

pb = calculate_payback(cf_project)                 # 非折现回收期
dpb = discounted_payback(cf_project, 0.10)         # 折现回收期
print(f'非折现回收期 = {pb:.2f} 年')                 # 2 + 30/35 ≈ 2.86 年
print(f'折现回收期 = {dpb:.2f} 年（折现后更慢）')     # 一定 ≥ 非折现

## 5. 多项目对比与 NPV Profile

In [ ]:
projects = {                                      # 三个备选项目（万元）
    '项目A': [-100, 30, 40, 35, 25],               # 课堂例题
    '项目B': [-150, 50, 55, 60, 40],               # 大投资大回收
    '项目C': [-80, 25, 30, 30, 20],                # 小投资稳回收
}

rows = []                                         # 结果收集
for name, cf in projects.items():                 # 遍历项目
    npv = calculate_npv(cf, 0.10)                 # NPV
    irr = calculate_irr_bisection(cf, 0.0, 0.50)  # IRR
    pbk = calculate_payback(cf)                   # 回收期
    rows.append({'项目': name, 'NPV(万元)': round(npv, 2),   # 汇总一行
                 'IRR': f'{irr:.1%}', '回收期(年)': round(pbk, 2)})

summary = pd.DataFrame(rows)                      # 转表格
summary                                           # 显示对比表

In [ ]:
# ===== NPV Profile：NPV 随折现率的变化曲线 =====
rates = np.linspace(0, 0.30, 100)                 # 折现率 0~30%
colors = ['#C49A6C', '#8B6F4E', '#4A7C9B']        # 三项目颜色

plt.figure(figsize=(12, 7))                        # 画布
for (name, cf), color in zip(projects.items(), colors):   # 遍历项目
    npvs = [calculate_npv(cf, r) for r in rates]   # 每个折现率的 NPV
    plt.plot(rates * 100, npvs, linewidth=2.5, label=name, color=color)   # 画曲线
    irr = calculate_irr_bisection(cf, 0.0, 0.50)  # 该项目 IRR
    plt.scatter([irr * 100], [0], color=color, s=80, zorder=5)   # 标出曲线与横轴交点 = IRR

plt.axhline(0, color='gray', alpha=0.6)           # 零线（NPV=0）
plt.axvline(10, color='#E74C3C', linestyle='--', alpha=0.7, label='要求回报率 10%')   # 资本成本竖线
plt.xlabel('折现率 r (%)')                         # x 轴
plt.ylabel('NPV（万元）')                          # y 轴
plt.title('NPV Profile：曲线与横轴交点即 IRR')      # 标题
plt.legend()                                      # 图例
plt.grid(True, alpha=0.3)                         # 网格
plt.show()                                        # 显示

In [ ]:
# ===== 敏感性分析：现金流变动对 NPV 的影响 =====
print('现金流敏感性（项目A，r=10%）：')              # 标题
for pct in [-20, -10, 0, 10, 20]:                 # 五档变动
    adjusted = [cf_project[0]] + [cf * (1 + pct/100) for cf in cf_project[1:]]   # 只有未来现金流变动
    npv_adj = calculate_npv(adjusted, 0.10)       # 重算 NPV
    print(f'  现金流 {pct:+d}% → NPV = {npv_adj:+.2f} 万元')   # 打印对比

## 6. 四方法对比与决策优先级

| 方法 | 规则 | 优点 | 缺点 |
|---|---|---|---|
| NPV | > 0 接受 | 衡量财富增加（黄金标准）| 折现率需估计 |
| IRR | > 要求回报率 | 直观百分比 | 非常规现金流可能多解 |
| ARR | > 目标 | 简单 | 忽略时间价值、用利润 |
| 回收期 | < 目标期 | 强调流动性 | 忽略期后现金流 |

**优先级：NPV > IRR > 回收期 > ARR**；互斥项目选 NPV 最大者。

## 7. 课程总结

八周课程一条主线：**数据 → 统计 → 关系 → 预测 → 短期决策 → 长期决策**。

- W1–W2 打好数据与统计地基（类型、抽样、描述统计）
- W3–W4 建立变量关系模型（成本函数、最小二乘回归）
- W5 学会时间维度预测（趋势 + 季节）
- W6–W7 短期决策（瓶颈分配、CVP、EOQ）
- W8 长期决策（货币时间价值、NPV/IRR）

决策思维四问：**数据类型对吗？相关成本找到了吗？瓶颈在哪？NPV 为正吗？**